# Rule-based & expert systems — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## Forward chaining from facts through rules
An expert system separates facts from if-then rules. These examples show one rule firing, a chain of rules, conflict resolution, certainty factors, and an explainable diagnosis.

In [ ]:
# 1) one rule fires when all antecedents are present
facts={'fever','cough'}; rule=( {'fever','cough'}, 'flu' ); fired=rule[0] <= facts
plt.figure(figsize=(5,3)); plt.bar(['fever','cough','flu inferred'], [1,1,fired], color='tab:green'); plt.title('IF fever and cough THEN flu')
assert fired is True

In [ ]:
# 2) forward chaining repeats until no new facts appear
rules=[({'fever','cough'},'flu'),({'flu'},'rest'),({'rest'},'hydrate')]; facts={'fever','cough'}; counts=[]
changed=True
while changed:
    changed=False
    for ant,cons in rules:
        if ant <= facts and cons not in facts: facts.add(cons); changed=True
    counts.append(len(facts))
plt.figure(figsize=(5,3)); plt.plot(counts,'-o'); plt.title('facts grow to a fixed point')
assert 'hydrate' in facts and counts[-1]==5

In [ ]:
# 3) conflict resolution can prefer the more specific rule
rules=[('general allergy',1,{'sneeze'},'allergy'),('specific flu',2,{'fever','cough'},'flu')]; facts={'fever','cough','sneeze'}
app=[r for r in rules if r[2] <= facts]; chosen=max(app, key=lambda r:r[1])
plt.figure(figsize=(6,3)); plt.bar([r[0] for r in app], [r[1] for r in app], color='tab:orange'); plt.title('specificity priority')
assert chosen[3]=='flu'

In [ ]:
# 4) certainty factors combine rule strengths
cf_fever=.8; cf_cough=.7; rule_cf=.9; flu_cf=min(cf_fever,cf_cough)*rule_cf
plt.figure(figsize=(5,3)); plt.bar(['fever','cough','flu'], [cf_fever,cf_cough,flu_cf], color='tab:blue'); plt.ylim(0,1); plt.title('min antecedent times rule strength')
assert abs(flu_cf-0.63)<1e-9

In [ ]:
# 5) explanation is the fired rule trace
evidence=['fever','cough']; conclusion='flu'; plt.figure(figsize=(6,2)); plt.plot([0,1,2],[0,0,0],'-o')
for i,t in enumerate(['fever+cough','rule R1','flu']): plt.text(i,.05,t,ha='center')
plt.axis('off'); plt.title('why flu? evidence -> rule -> conclusion')
assert evidence==['fever','cough'] and conclusion=='flu'